# TTE map review: western Alaska and the Brooks Range

Maps the TTE map tiles that DPS (`run_tte_map`) published in the MAAP DPS STAC catalog, through the
MAAP TiTiler (titiler-pgstac):

* **TTE classes** (GEE palette), over the **TTE domain** (gray); **tree canopy cover** and
  **abruptness** layers are in the layer list.
* **Click** anywhere for the pixel's class, tree cover, abruptness and zone (TiTiler point query).
* **Hover** a tile of the grid for its STAC metadata: tree cover source and year, classified area and
  share of the domain, mean tree cover and abruptness, the share of each structure group, code version,
  DPS job tag.

Needs the tiles to be published first: run `dps/DPS_tte_map.ipynb` (test tile, then the review set)
and wait for ingestion. Map layers below zoom 5 are not drawn (TiTiler is slow over many tiles).
The map is also saved as a standalone HTML file that works in any browser.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'lib'))   # ../lib (run from notebooks/)
sys.path.insert(0, os.path.expanduser('~/code/tte/lib'))
import pandas as pd
import tte_review as tr

## Settings
* `COLLECTION`: where DPS published the tiles, `{user}__{algorithm}__{version}`.
* `TCC_LABEL`: which run to show (items record `tte:tcc_label`, e.g. `hansen2010`, `metachm2m`); `None` shows all.
* `REGIONS`: lon/lat boxes to show.

In [ ]:
COLLECTION = 'montesano__run_tte_map__tte_map_v1'
TCC_LABEL = 'hansen2010'
REGIONS = {'western Alaska': (-166.0, 63.5, -155.0, 69.0), 'Brooks Range': (-155.0, 66.5, -141.0, 69.0)}
BBOX = (min(b[0] for b in REGIONS.values()), min(b[1] for b in REGIONS.values()),
        max(b[2] for b in REGIONS.values()), max(b[3] for b in REGIONS.values()))
OUT_HTML = os.path.expanduser(f'~/my-public-bucket/tte/review/TTE_review_{COLLECTION}_{TCC_LABEL}.html')
print(f'{tr.BROWSER}/collections/{COLLECTION}')

## 1. The published tiles (STAC items)

In [ ]:
grid = tr.items_gdf(COLLECTION, bbox=BBOX, tcc_label=TCC_LABEL)
print(len(grid), 'tiles published for', TCC_LABEL)
if len(grid):
    cols = [c for c in ['tile_num', 'tcc_label', 'classified_km2', 'classified_share_of_domain', 'mean_tcc_domain',
                        'share_sparse', 'share_open', 'share_intermediate_closed', 'share_nonforest_edge', 'maap-dps:tag']
            if c in grid]
    display(grid[cols].sort_values('tile_num').round(3).head(20))
else:
    print('No tiles yet: run the DPS jobs (dps/DPS_tte_map.ipynb) and wait for STAC ingestion.')

## 2. Map

In [ ]:
SEARCH_ID = tr.register_search(COLLECTION, bbox=BBOX, tcc_label=TCC_LABEL)   # a TiTiler mosaic of these items
m = tr.review_map(SEARCH_ID, grid=grid, center=[67.3, -153], zoom=6)
os.makedirs(os.path.dirname(OUT_HTML), exist_ok=True)
m.save(OUT_HTML)
print('saved', OUT_HTML)
m